In [35]:
#!pip install geopandas
# !pip install shapely
import geopandas as gpd
from shapely import wkt
import pandas as pd

## Fazendo georreferenciamento

O código abaixo foi, em sua maior parte, gerado po IA, pois nenhum dos membros do grupo tinha conhecimento sobre as tecnologias necessárias. O código basicamente pega uma database disponibilizada pelo ISPdados que contém os limites de cada região de segurança (cisp) e associa com o dataset da smtr que contém os trajetos completos de todas as linhas. Juntando os dois dados, obtemos a distância que cada linha de ônibus percorre em cada cisp. 

In [36]:
cisps = gpd.read_file('../datasets/cisp_geo_data/lm_cisp_bd.shp')
cisps

,cisp,aisp,shape_Leng,shape_Area,AREA_GEO,geometry
0,120,35,1.595433,0.082321,9.375473e+08,"POLYGON ((-42.18687 -22.55548, -42.18733 -22.5..."
1,127,25,0.697487,0.006180,7.027806e+07,"MULTIPOLYGON (((-41.90082 -22.78264, -41.90079..."
2,151,11,1.752923,0.081815,9.334145e+08,"POLYGON ((-42.52555 -22.16087, -42.5006 -22.18..."
3,107,38,1.209676,0.050835,5.805239e+08,"POLYGON ((-43.34665 -22.00375, -43.34603 -22.0..."
4,123,32,2.118860,0.106640,1.216847e+09,"MULTIPOLYGON (((-41.97224 -22.1428, -41.97201 ..."
...,...,...,...,...,...,...
132,76,12,0.338630,0.000828,9.410025e+06,"MULTIPOLYGON (((-43.11375 -22.8646, -43.11358 ..."
133,81,12,0.593683,0.005718,6.495636e+07,"MULTIPOLYGON (((-42.98891 -22.89112, -42.9887 ..."
134,9,2,0.204071,0.000603,6.852825e+06,"POLYGON ((-43.16851 -22.91435, -43.16854 -22.9..."
135,1,5,0.306591,0.000308,3.503511e+06,"MULTIPOLYGON (((-43.17829 -22.89257, -43.1795 ..."


In [37]:
def carregar_wkt(texto):
    try:
        return wkt.loads(texto)
    except:
        return None

In [38]:
df = pd.read_csv("../datasets/shapes_geom.csv")
df.head()

,feed_version,feed_start_date,feed_end_date,shape_id,shape,shape_distance,start_pt,end_pt,versao_modelo
0,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961
1,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8
2,2025-11-08,2025-11-08,2025-11-18,35bu,"MULTILINESTRING((-43.17754 -22.90107, -43.1775...",33079.0,POINT(-43.17754 -22.90107),POINT(-43.39397 -22.95665),2a53b9c61f4b76fe07761a33083c714f9c35bcd0
3,2025-10-04,2025-10-04,2025-10-11,iz18,"LINESTRING(-43.19987 -22.93997, -43.19975 -22....",12591.1,POINT(-43.19987 -22.93997),POINT(-43.2238 -22.98097),40c2bc268bdc2e22bc2474d99bca38dcb9ea4c40
4,2024-03-11,2024-03-11,2024-03-17,2ibq,"LINESTRING(-43.621884 -22.968783, -43.62189 -2...",24430.2,POINT(-43.621884 -22.968783),POINT(-43.4637 -22.87668),201d79faee763526a030ff998bebea9782efe961


In [39]:
df['geometria'] = df['shape'].apply(carregar_wkt)

In [40]:
gdf_rotas = gpd.GeoDataFrame(df, geometry='geometria', crs="EPSG:4326")

In [41]:
gdf_rotas_m = gdf_rotas.to_crs(epsg=31983)
cisps_m = cisps.to_crs(epsg=31983)

In [42]:
recortes = gpd.overlay(gdf_rotas_m, cisps_m, how='intersection')
recortes['distancia_m'] = recortes.geometry.length
recortes['distancia_km'] = recortes['distancia_m'] / 1000

In [43]:
recortes = recortes.drop_duplicates(subset=['shape_id', 'cisp', 'distancia_km'])

In [44]:
recortes

,feed_version,feed_start_date,feed_end_date,shape_id,shape,shape_distance,start_pt,end_pt,versao_modelo,cisp,aisp,shape_Leng,shape_Area,AREA_GEO,geometry,distancia_m,distancia_km
0,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961,19,6,0.420198,0.003561,4.044724e+07,"LINESTRING (681305.403 7463828.277, 681365.041...",12144.652919,12.144653
1,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961,16,31,0.498799,0.005553,6.304382e+07,"LINESTRING (675016.345 7457726.047, 675011.161...",12306.245858,12.306246
2,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8,4,5,0.097140,0.000334,3.792617e+06,"LINESTRING (685182.363 7465695.422, 685483.042...",721.359239,0.721359
3,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8,17,4,0.253581,0.000981,1.114670e+07,"LINESTRING (681997.629 7465170.958, 682009.69 ...",101.148169,0.101148
4,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8,26,3,0.146193,0.000864,9.817335e+06,"LINESTRING (675357.735 7466447.699, 675366.199...",3531.868455,3.531868
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92385,2023-09-01,2023-09-01,2023-09-15,n4ts,"MULTILINESTRING((-43.36642 -22.80665, -43.3663...",41401.8,POINT(-43.36642 -22.80665),POINT(-43.1766 -22.91606),201d79faee763526a030ff998bebea9782efe961,31,41,0.219618,0.001248,1.419032e+07,"MULTILINESTRING ((665359.195 7475977.318, 6653...",6987.015808,6.987016
92386,2023-09-01,2023-09-01,2023-09-15,n4ts,"MULTILINESTRING((-43.36642 -22.80665, -43.3663...",41401.8,POINT(-43.36642 -22.80665),POINT(-43.1766 -22.91606),201d79faee763526a030ff998bebea9782efe961,9,2,0.204071,0.000603,6.852825e+06,"LINESTRING (687019.719 7464787.102, 687024.966...",469.075471,0.469075
92387,2023-09-01,2023-09-01,2023-09-15,n4ts,"MULTILINESTRING((-43.36642 -22.80665, -43.3663...",41401.8,POINT(-43.36642 -22.80665),POINT(-43.1766 -22.91606),201d79faee763526a030ff998bebea9782efe961,1,5,0.306591,0.000308,3.503511e+06,"LINESTRING (685866.545 7465923.991, 685965.176...",818.300786,0.818301
92425,2023-10-17,2023-10-17,2023-10-23,ejro,"MULTILINESTRING((-43.18086 -22.90192, -43.1791...",30258.2,POINT(-43.18086 -22.90192),POINT(-43.35484 -22.85999),201d79faee763526a030ff998bebea9782efe961,4,5,0.097140,0.000334,3.792617e+06,"MULTILINESTRING ((685844.952 7465972.371, 6853...",1937.820484,1.937820


In [45]:
resultado = recortes.groupby('shape_id').apply(
    lambda x: list(zip(x['cisp'], round(x['distancia_km'], 2)))
).reset_index()

resultado.columns = ['id_linha', 'distancia_por_cisp']
resultado['distancia_por_cisp'] = resultado['distancia_por_cisp'].apply(lambda x: list(set(x)))

C:\Users\rhena\AppData\Local\Temp\ipykernel_13416\3127763427.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  resultado = recortes.groupby('shape_id').apply(


In [46]:
resultado

,id_linha,distancia_por_cisp
0,006j,"[(4, 0.13), (18, 1.59), (14, 2.62), (7, 1.24),..."
1,006j_0,"[(17, 1.08), (18, 1.1), (7, 0.62), (10, 0.6), ..."
2,006j_1,"[(4, 0.13), (10, 0.68), (17, 1.16), (6, 2.97),..."
3,01gk,"[(34, 8.5)]"
4,01o6,"[(4, 1.21), (9, 0.08), (1, 3.9), (4, 0.41), (4..."
...,...,...
1909,zubh,"[(18, 0.05), (40, 1.52), (38, 3.48), (35, 0.36..."
1910,zx4l,"[(33, 13.45), (34, 2.07)]"
1911,zy4z,"[(32, 2.11), (41, 0.61)]"
1912,zy58,"[(23, 0.38), (6, 1.46), (44, 2.44), (4, 0.72),..."


In [51]:
resultado.loc[663]["distancia_por_cisp"]

[(36, 11.8), (43, 1.34), (43, 1.37), (36, 6.28)]

## Tratamento

Após o bloco de vibe code me dar as rotas quebradas por cisp, resta colocá-los em nomes mais legíveis para facilitar o uso.

### Adicionar informações de linhas


In [48]:
aux = pd.read_csv("../datasets/trips.csv")
aux = aux[["route_id","shape_id"]].drop_duplicates()
mid = pd.merge(aux,  resultado,right_on="id_linha", left_on="shape_id", how="inner")
mid = mid.drop("id_linha", axis=1).drop_duplicates(subset=["shape_id","route_id"])
mid.head()
len(mid)

1853

In [61]:
linhas_df = pd.read_csv("../datasets/routes.csv")
linhas_df = linhas_df[["route_id","route_short_name","route_long_name"]].drop_duplicates(subset="route_id")
df_cool = pd.merge(mid,linhas_df,on="route_id")
df_cool

,route_id,shape_id,distancia_por_cisp,route_short_name,route_long_name
0,E2336AAA0A,ab43,"[(40, 1.52), (5, 2.06), (27, 1.91), (1, 0.17),...",2336,Campo Grande - Castelo
1,O0629AAA0A,ue02,"[(21, 2.13), (25, 4.88), (27, 5.54), (38, 1.08...",629,Irajá - Saens Peña
2,O0774AAA0A,i9sp,"[(38, 7.94), (27, 4.65), (29, 4.88), (27, 4.64...",774,Madureira - Jardim América
3,O0112AAA0A,O0112AAA0AVDU03,"[(6, 5.81), (18, 0.05), (15, 4.91), (4, 1.31),...",112,Terminal Gentileza - Alto Gávea
4,O0343AAA0A,j7v9,"[(18, 0.05), (26, 2.32), (1, 1.8), (4, 1.31), ...",343,Jardim Oceânico - Candelária
...,...,...,...,...,...
1848,O0777AAV0A,nsvy,"[(39, 1.82), (40, 2.49), (33, 11.57), (29, 5.0...",SV777,Padre Miguel - Madureira
1849,O0355AAE0A,j34o,"[(9, 3.3), (17, 2.89), (12, 1.47), (22, 4.95),...",SE355,Madureira - Enseada de Botafogo
1850,O0774AAV0A,aqav,"[(38, 4.26), (29, 3.78), (27, 5.97)]",SV774,Madureira - Jardim América
1851,O0104AAA0A,ii9t,"[(18, 0.05), (10, 0.78), (5, 2.83), (17, 0.67)...",104,São Conrado - Terminal Gentileza


In [65]:
n = 122
print(df_cool.loc[n])
print(sorted(df_cool.loc[n]["distancia_por_cisp"]))

route_id                                O0783AAP0A
shape_id                                      se4q
distancia_por_cisp        [(28, 5.67), (30, 3.54)]
route_short_name                             SP783
route_long_name       Marechal Hermes - Praça Seca
Name: 122, dtype: object
[(28, 5.67), (30, 3.54)]


In [ ]:
### adicionar 